# 01 · Homografía aplicada a vídeo con propagación por movimiento de cámara

**Objetivo:** aplicar el pipeline de proyección (detección + keypoints + homografía) a un vídeo completo, generando una pizarra táctica 2D animada sincronizada. En los frames sin keypoints suficientes, la homografía se **propaga** estimando el movimiento de cámara, en lugar de congelar la última matriz válida.

**Contexto:** en este material aproximadamente la mitad de los frames no alcanzan los 4 keypoints necesarios para calcular la homografía (planos abiertos del centro del campo, donde solo se ve parte del círculo central). Congelar la última matriz deja a los jugadores descuadrados durante los paneos. En su lugar, se estima el desplazamiento de la cámara entre frames por flujo óptico (Lucas-Kanade sobre features Shi-Tomasi) y se compone con la última homografía real. RANSAC descarta los puntos que se mueven distinto al fondo (los jugadores), de modo que la estimación refleja solo el movimiento de cámara. Cuando reaparecen keypoints, la homografía se reancla a la verdad de campo, cortando la deriva acumulada.

**Entrada:** vídeo de prueba, modelos `keypoints_base.pt` y `best_v4.pt`.

**Salida:** vídeo de la pizarra 2D animada en `outputs/`.

In [1]:
import numpy as np
import cv2
from ultralytics import YOLO
from pathlib import Path
from collections import deque

REPO_ROOT = Path("../..").resolve()

# Modelos
kp_model = YOLO(str(REPO_ROOT / "models" / "keypoints_base.pt"))
detector = YOLO(str(REPO_ROOT / "models" / "best_v4.pt"))

# Coordenadas reales de los 24 keypoints (metros)
KP_REALES = np.array([
    (0.0,0.0),(0.0,68.0),(105.0,0.0),(105.0,68.0),
    (16.5,13.84),(16.5,54.16),(88.5,13.84),(88.5,54.16),
    (5.5,24.84),(5.5,43.16),(99.5,24.84),(99.5,43.16),
    (11.0,34.0),(94.0,34.0),(52.5,34.0),(52.5,0.0),(52.5,68.0),
    (0.0,13.84),(0.0,54.16),(105.0,13.84),
    (0.0,30.34),(0.0,37.66),(105.0,30.34),(105.0,37.66),
], dtype=np.float32)

EXCLUIR = {14, 15, 16}   # centro, media_inf, media_sup (poco fiables)
CONF_KP = 0.5
CONF_DET = 0.3

print("Modelos cargados")

Modelos cargados


In [2]:
# RUTA DEL VÍDEO DE PRUEBA — ajusta a tu archivo
VIDEO_IN = REPO_ROOT / "data" / "raw" / "A1606b0e6_0 (62).mp4"

print(f"Vídeo existe: {VIDEO_IN.exists()}")
if VIDEO_IN.exists():
    cap = cv2.VideoCapture(str(VIDEO_IN))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f"FPS: {fps}, Frames: {total}, Resolución: {w}x{h}")
    print(f"Duración: {total/fps:.1f} s")

Vídeo existe: True
FPS: 25.0, Frames: 749, Resolución: 1920x1080
Duración: 30.0 s


In [3]:
def calcular_homografia(frame):
    """Detecta keypoints en un frame y devuelve la matriz H (o None si no hay suficientes)."""
    res = kp_model(frame, verbose=False, conf=0.25)
    r = res[0]
    if r.keypoints is None:
        return None
    kps_xy = r.keypoints.xy[0].cpu().numpy()
    kps_conf = r.keypoints.conf[0].cpu().numpy()

    idx = [i for i in range(24) if kps_conf[i] > CONF_KP and i not in EXCLUIR]
    if len(idx) < 4:
        return None

    src = kps_xy[idx]
    dst = KP_REALES[idx]
    H, _ = cv2.findHomography(src, dst, cv2.RANSAC, 3.0)
    return H

def proyectar_jugadores(frame, H):
    """Detecta jugadores y proyecta sus pies al campo. Devuelve lista de (x, y, clase)."""
    res = detector(frame, verbose=False, conf=CONF_DET)
    rd = res[0]
    if len(rd.boxes) == 0:
        return []
    boxes = rd.boxes.xyxy.cpu().numpy()
    clases = rd.boxes.cls.cpu().numpy().astype(int)

    pies = np.array([[[ (x1+x2)/2, y2 ]] for (x1,y1,x2,y2) in boxes], dtype=np.float32)
    proyectados = []
    for p, c in zip(pies, clases):
        pt = cv2.perspectiveTransform(p.reshape(1,1,2), H)[0][0]
        x, y = pt
        # Filtro de cordura: descartar fuera del campo
        if -3 <= x <= 108 and -3 <= y <= 71:
            proyectados.append((x, y, c))
    return proyectados

print("Funciones definidas")

Funciones definidas


## Procesamiento del vídeo con propagación de homografía

Se recorre el vídeo frame a frame. Si el frame tiene ≥4 keypoints utilizables, se calcula la homografía real y la matriz vigente se **reancla** a ella (con media móvil para suavizar el temblor). Si no, se estima el movimiento de cámara desde el frame anterior por flujo óptico y se **compone** con la última matriz vigente, propagando la calibración a través del paneo. Una guarda descarta movimientos físicamente imposibles (cortes de plano) cayendo a la última matriz válida. Los contadores finales permiten verificar cuántos frames usaron cada vía.

In [4]:
# ---- Procesamiento del vídeo con PROPAGACIÓN de homografía por movimiento de cámara ----
# En frames sin >=4 keypoints, en lugar de congelar la última H, estima el movimiento de
# cámara (flujo óptico LK sobre features Shi-Tomasi + homografía inter-frame con RANSAC) y
# lo compone con la última H real. RANSAC descarta los puntos que se mueven distinto al
# fondo (jugadores), de modo que la estimación refleja solo el movimiento de cámara.

PROCESAR_CADA = 1
VENTANA_SUAVIZADO = 5          # media móvil sobre las H REALES (anti-temblor)

# Parámetros de flujo óptico
FEAT_PARAMS = dict(maxCorners=400, qualityLevel=0.01, minDistance=7, blockSize=7)
LK_PARAMS = dict(winSize=(21, 21), maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
MIN_PTS_FLUJO = 12             # mínimo de puntos emparejados para fiarse del movimiento
MAX_DESPL_MEDIO = 80.0         # px: desplazamiento medio máximo plausible entre 2 frames
                              # (paneo suave); por encima = corte de plano -> guarda

cap = cv2.VideoCapture(str(VIDEO_IN))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

historial_H = deque(maxlen=VENTANA_SUAVIZADO)  # solo H REALES, para suavizar el reanclaje
H_propagada = None             # H vigente (real reanclada o propagada)
gris_prev = None               # frame anterior en gris, para el flujo
frames_data = []

n_real = 0       # frames con homografía calculada por keypoints
n_propag = 0     # frames con homografía propagada por movimiento
n_guarda = 0     # frames donde la propagación se descartó (movimiento absurdo / sin puntos)
n_sin_nada = 0   # frames sin ninguna H disponible (arranque antes del primer keypoint)

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % PROCESAR_CADA != 0:
        frame_idx += 1
        continue

    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    H_real = calcular_homografia(frame)   # None si <4 keypoints

    if H_real is not None:
        # Frame con keypoints: reancla. La H vigente vuelve a la verdad de campo.
        historial_H.append(H_real)
        H_suave = np.mean(np.array(historial_H), axis=0)
        H_propagada = H_suave
        n_real += 1
    else:
        # Frame sin keypoints: propagar la última H vigente con el movimiento de cámara.
        movimiento_ok = False
        if H_propagada is not None and gris_prev is not None:
            p0 = cv2.goodFeaturesToTrack(gris_prev, mask=None, **FEAT_PARAMS)
            if p0 is not None and len(p0) >= MIN_PTS_FLUJO:
                p1, st, err = cv2.calcOpticalFlowPyrLK(gris_prev, gris, p0, None, **LK_PARAMS)
                if p1 is not None:
                    buenos = st.ravel() == 1
                    a, b = p0[buenos].reshape(-1, 2), p1[buenos].reshape(-1, 2)
                    if len(a) >= MIN_PTS_FLUJO:
                        despl_medio = np.linalg.norm(b - a, axis=1).mean()
                        if despl_medio <= MAX_DESPL_MEDIO:
                            # Homografía imagen->imagen entre frame anterior y actual.
                            # RANSAC descarta puntos de jugadores (se mueven distinto al fondo).
                            H_mov, inl = cv2.findHomography(a, b, cv2.RANSAC, 3.0)
                            if H_mov is not None and inl is not None and inl.sum() >= MIN_PTS_FLUJO:
                                # Componer: H_actual->campo = H_prev->campo · (imagen_actual->imagen_prev)
                                H_propagada = H_propagada @ np.linalg.inv(H_mov)
                                movimiento_ok = True
        if movimiento_ok:
            n_propag += 1
        elif H_propagada is not None:
            n_guarda += 1          # se mantiene la última H vigente sin propagar (guarda)
        else:
            n_sin_nada += 1

    # Proyectar jugadores con la H vigente (real reanclada o propagada)
    if H_propagada is not None:
        jugadores = proyectar_jugadores(frame, H_propagada)
    else:
        jugadores = []
    frames_data.append(jugadores)

    gris_prev = gris
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"Procesados {frame_idx}/{total} frames...")

cap.release()
print(f"\nCompletado sobre {len(frames_data)} frames:")
print(f"  Homografía REAL (keypoints):        {n_real}")
print(f"  Homografía PROPAGADA (movimiento):  {n_propag}")
print(f"  Guarda (última H, sin propagar):    {n_guarda}")
print(f"  Sin ninguna H (arranque):           {n_sin_nada}")
print(f"  -> Frames con proyección utilizable: {n_real + n_propag + n_guarda} de {len(frames_data)}")

Procesados 50/749 frames...
Procesados 100/749 frames...
Procesados 150/749 frames...
Procesados 200/749 frames...
Procesados 250/749 frames...
Procesados 300/749 frames...
Procesados 350/749 frames...
Procesados 400/749 frames...
Procesados 450/749 frames...
Procesados 500/749 frames...
Procesados 550/749 frames...
Procesados 600/749 frames...
Procesados 650/749 frames...
Procesados 700/749 frames...

Completado sobre 749 frames:
  Homografía REAL (keypoints):        373
  Homografía PROPAGADA (movimiento):  376
  Guarda (última H, sin propagar):    0
  Sin ninguna H (arranque):           0
  -> Frames con proyección utilizable: 749 de 749


In [5]:
from matplotlib.patches import Circle, Rectangle, Arc

def dibujar_campo(ax):
    """Dibuja un campo de fútbol 2D a escala (105x68 m) sobre el eje dado."""
    ax.add_patch(Rectangle((0, 0), 105, 68, color="#3a7d34", zorder=0))
    lc, lw = "white", 2
    ax.add_patch(Rectangle((0, 0), 105, 68, fill=False, edgecolor=lc, linewidth=lw))
    ax.plot([52.5, 52.5], [0, 68], color=lc, linewidth=lw)
    ax.add_patch(Circle((52.5, 34), 9.15, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((52.5, 34), 0.4, color=lc))
    ax.add_patch(Rectangle((0, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((0, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((11, 34), 0.4, color=lc))
    ax.add_patch(Arc((11, 34), 18.3, 18.3, theta1=-53, theta2=53, color=lc, linewidth=lw))
    ax.add_patch(Rectangle((88.5, 13.84), 16.5, 40.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Rectangle((99.5, 24.84), 5.5, 18.32, fill=False, edgecolor=lc, linewidth=lw))
    ax.add_patch(Circle((94, 34), 0.4, color=lc))
    ax.add_patch(Arc((94, 34), 18.3, 18.3, theta1=127, theta2=233, color=lc, linewidth=lw))
    ax.set_xlim(-5, 110)
    ax.set_ylim(-5, 73)
    ax.set_aspect("equal")
    ax.axis("off")

print("Función dibujar_campo definida")

Función dibujar_campo definida


In [6]:
# Generar el vídeo de la pizarra animada
import matplotlib
matplotlib.use("Agg")  # backend sin ventana, para renderizar a archivo
import matplotlib.pyplot as plt

OUTPUT_VIDEO = REPO_ROOT / "outputs" / "pizarra_animada.mp4"
OUTPUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)

colores = {0: "yellow", 1: "red", 2: "black"}

# Crear el writer de vídeo (tamaño fijo)
PIZARRA_W, PIZARRA_H = 1100, 800
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
fps_salida = fps / PROCESAR_CADA
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps_salida, (PIZARRA_W, PIZARRA_H))

for i, jugadores in enumerate(frames_data):
    fig, ax = plt.subplots(figsize=(11, 8), dpi=100)
    dibujar_campo(ax)

    for (x, y, c) in jugadores:
        ax.scatter(x, y, c=colores.get(c, "blue"), s=120,
                   edgecolors="white", linewidth=1.5, zorder=5)

    # Convertir la figura de matplotlib a imagen para el vídeo
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img = buf.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    img = cv2.resize(img, (PIZARRA_W, PIZARRA_H))
    writer.write(img)
    plt.close(fig)

    if i % 50 == 0:
        print(f"Renderizados {i}/{len(frames_data)} frames del vídeo...")

writer.release()
print(f"\nVídeo guardado en: {OUTPUT_VIDEO}")

Renderizados 0/749 frames del vídeo...
Renderizados 50/749 frames del vídeo...
Renderizados 100/749 frames del vídeo...
Renderizados 150/749 frames del vídeo...
Renderizados 200/749 frames del vídeo...
Renderizados 250/749 frames del vídeo...
Renderizados 300/749 frames del vídeo...
Renderizados 350/749 frames del vídeo...
Renderizados 400/749 frames del vídeo...
Renderizados 450/749 frames del vídeo...
Renderizados 500/749 frames del vídeo...
Renderizados 550/749 frames del vídeo...
Renderizados 600/749 frames del vídeo...
Renderizados 650/749 frames del vídeo...
Renderizados 700/749 frames del vídeo...

Vídeo guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\outputs\pizarra_animada.mp4
